# Tie-line levelling

Survey 2 is flown along parallel lines, survey 3 (the tie-line) crosses them. Every track is detrended,
the tie-line is matched to the survey at the crossings and the residuals are removed from the survey.
All processing lives in `src/tieline.py`; this notebook only plots.

The grids use PyMEL (`PyMEL2.0/`), which calls `nnbathy`. On macOS `nnbathy` has to be on the PATH.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

import tieline as tl
from dataloader import load_survey, load_object_locations
from paths import FIGURE_DIR

In [ ]:
SAVE_FIGURES = False   # write the figures to FIGURE_DIR

survey_raw = load_survey(tl.SURVEY_INDEX, tl.SURVEY_SPEED)
tieline_raw = load_survey(tl.TIELINE_INDEX, tl.SURVEY_SPEED)
objects = load_object_locations()

survey, survey_slopes = tl.detrend_tracks(survey_raw, exclude_tracks=tl.EXCLUDED_TRACKS)
tieline, tieline_slopes = tl.detrend_tracks(tieline_raw)
survey, matched = tl.tieline_correction(survey, tieline)

print("Survey tracks:", sorted(survey["Track_number"].unique()))
print("Matched tie-line samples per sensor:", {s: len(m) for s, m in matched.items()})

## Detrended tracks

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
axes[0].scatter(tieline_raw["Longitude [m]"], tieline_raw["Latitude [m]"], s=1)
axes[0].set_title("Tie-line flight")
for ax, data, title in [(axes[1], survey, "Survey, detrended"), (axes[2], tieline, "Tie-line, detrended")]:
    sc = ax.scatter(data["Longitude [m]"], data["Latitude [m]"], c=data["detrended"], cmap="magma", s=1)
    fig.colorbar(sc, ax=ax, label="Detrended field [nT]")
    ax.set_title(title)
for ax in axes:
    ax.set_xlabel("Longitude [m]")
    ax.set_ylabel("Latitude [m]")
plt.tight_layout()
plt.show()

In [ ]:
magma = plt.get_cmap("magma")

plt.figure(figsize=(12, 8))
plt.scatter(survey["Longitude [m]"], survey["Latitude [m]"], color=magma(0.2), s=1, label=f"Survey {tl.SURVEY_INDEX}", alpha=0.7)
plt.scatter(tieline["Longitude [m]"], tieline["Latitude [m]"], color=magma(0.8), s=1, label="Tieline", alpha=0.7)
plt.xlabel("Longitude [m]", fontsize=20, labelpad=15)
plt.ylabel("Latitude [m]", fontsize=20, labelpad=15)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.legend(markerscale=5, fontsize=16)
plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
plt.axis("equal")
plt.tight_layout()
plt.show()

## Crossings

In [ ]:
sensor = 5
crossings = matched[sensor]

fig, ax = plt.subplots(figsize=(14, 10))
ax.scatter(survey["Longitude [m]"], survey["Latitude [m]"], color="lightgray", s=1, label="Survey", alpha=0.6)
ax.scatter(crossings["Longitude [m]"], crossings["Latitude [m]"], color="green", s=8, label="Matched Tie-line Points", alpha=0.9)
ax.set_xlim([survey["Longitude [m]"].min() - 15, survey["Longitude [m]"].max() + 15])
ax.set_ylim([survey["Latitude [m]"].min() - 5, survey["Latitude [m]"].max() + 5])
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Longitude [m]", fontsize=20, labelpad=15)
ax.set_ylabel("Latitude [m]", fontsize=20, labelpad=15)
ax.set_title(f"Survey and Matched Tie-line Points (Sensor {sensor})", fontsize=22, pad=20)
ax.legend(markerscale=4, fontsize=16, loc="best")
ax.tick_params(axis="both", labelsize=16)
ax.grid(True, linestyle="--", linewidth=0.7, alpha=0.5)
plt.tight_layout()

if SAVE_FIGURES:
    FIGURE_DIR.mkdir(exist_ok=True)
    plt.savefig(FIGURE_DIR / f"survey_tieline_matched_sensor{sensor}.png", dpi=300)
plt.show()

## Before and after detrending (2 Hz lowpass)

In [ ]:
sns.set_theme(style="whitegrid")
palette = sns.color_palette("pastel")

fig, axes = plt.subplots(5, 2, figsize=(14, 12))
fig.subplots_adjust(hspace=0.5, wspace=0.3)
for sensor, res in tl.before_after(survey).items():
    for ax, key, title in [(axes[sensor - 1, 0], "before", "Before Correction"),
                           (axes[sensor - 1, 1], "after", "After Correction")]:
        ax.plot(res[key], color=palette[sensor - 1], linewidth=1.5)
        ax.set_title(f"Sensor {sensor} | {title}", fontsize=16, loc="left")
        ax.grid(True, linestyle="--", alpha=0.4)
        ax.tick_params(labelsize=14)
        ax.text(0.98, 0.95, f"σ = {res['std_' + key]:.2f}", transform=ax.transAxes,
                fontsize=12, ha="right", va="top", color="black")

fig.supxlabel("Sample Index", fontsize=18, y=0.05)
fig.supylabel("Field [nT]", fontsize=18, x=0.05)
plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95])
plt.show()

## Levelling residuals

In [ ]:
fits = tl.residual_fits(survey, laplace_sensors=(1, 2, 3, 4))
lower, upper = -5, 20
x = np.linspace(lower, upper, 200)

fig, axes = plt.subplots(1, 5, figsize=(22, 4), sharey=True)
for ax, (sensor, res) in zip(axes, fits.items()):
    ax.hist(res["residuals"], bins=200, color="skyblue", alpha=1, density=True, edgecolor="none")
    if res["dist"].name == "laplace":
        loc, scale = res["params"]
        ax.plot(x, res["dist"].pdf(x, loc, scale), "r--", linewidth=2, label=f"Laplace fit\nloc={loc:.2f}, scale={scale:.2f}")
    else:
        mu, sigma = res["params"]
        ax.plot(x, res["dist"].pdf(x, mu, sigma), "g--", linewidth=2, label=f"Gaussian fit\nμ={mu:.2f}, σ={sigma:.2f}")
    ax.set_xlim(lower, upper)
    ax.set_title(f"Sensor {sensor}")
    ax.set_xlabel("Residuals [nT]")
    ax.legend(fontsize=10)
    ax.grid(True, linestyle="--", alpha=1)
axes[0].set_ylabel("Density")

laplace_sensors = [s for s, r in fits.items() if r["dist"].name == "laplace"]
gauss_sensors = [s for s, r in fits.items() if r["dist"].name != "laplace"]
plt.suptitle(f"Residuals: Laplace ({', '.join(map(str, laplace_sensors))}) and Gaussian ({', '.join(map(str, gauss_sensors))}) Fits",
             fontsize=16, y=1.05)
plt.tight_layout()
plt.show()

## Grids per sensor

In [ ]:
grids = tl.sensor_grids(survey)
zooms = tl.zoom_grids(survey)

In [ ]:
box = tl.ZOOM_BOX
objects_zoom = objects[tl.in_box(objects, box)]

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(5, 2, figsize=(20, 40))
fig.subplots_adjust(hspace=0.5, wspace=0.35)

for sensor in range(1, 6):
    for ax, kind, title in [(axes[sensor - 1, 0], "raw", "Raw Field"), (axes[sensor - 1, 1], "corrected", "Corrected Field")]:
        grid = tl.mask_polygon(grids[sensor][kind], tl.SURVEY_POLYGON)
        sc = ax.scatter(grid["lon"], grid["lat"], c=grid["val"], cmap="magma", s=20, marker="s")
        ax.scatter(objects["Longitude [m]"], objects["Latitude [m]"], color="green", marker="*", s=150)
        ax.set_title(f"Sensor {sensor} | {title}", fontsize=22, loc="left")
        ax.tick_params(labelsize=18)
        ax.grid(True, linestyle="--", alpha=0.3)
        cbar = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=18)
        cbar.set_label("Field [nT]", fontsize=18)

    zoom = zooms[sensor]
    if zoom is not None:
        ax = axes[sensor - 1, 0]
        axins = inset_axes(ax, width="35%", height="35%", loc="upper right", borderpad=1)
        axins.scatter(zoom["lon"], zoom["lat"], c=zoom["val"], cmap="magma", s=1.2, marker="s")
        axins.scatter(objects_zoom["Longitude [m]"], objects_zoom["Latitude [m]"], color="green", marker="*", s=60)
        axins.set_xlim(box["lon_min"], box["lon_max"])
        axins.set_ylim(box["lat_min"], box["lat_max"])
        axins.set_xticks([])
        axins.set_yticks([])
        ax.add_patch(plt.Rectangle((box["lon_min"], box["lat_min"]), box["lon_max"] - box["lon_min"],
                                   box["lat_max"] - box["lat_min"], fill=False, edgecolor="blue", linewidth=2, linestyle="--"))
        mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.5", linestyle="dashed")

fig.text(0.5, 0.02, "Longitude [m]", ha="center", va="center", fontsize=24)
fig.text(0.02, 0.5, "Latitude [m]", ha="center", va="center", rotation="vertical", fontsize=24)
plt.tight_layout(rect=[0.06, 0.06, 0.95, 0.97])
plt.show()

## All sensors, detrended

In [ ]:
grid_all = tl.all_sensor_grid(survey)
grid_zoom = tl.anomaly_grid(survey, tl.ANOMALY_POLYGON, cell_size=0.025)

In [ ]:
vmin, vmax = -20, 60
polygon = tl.ANOMALY_POLYGON

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(18, 10))
fig.subplots_adjust(wspace=0.25)

for ax, grid, objs, s, title in [
    (axes[0], grid_all, objects, 12.5, "All Sensors | Detrended Field (clean cut)"),
    (axes[1], grid_zoom, tl.objects_in(objects, polygon), 8, "Zoom on Anomaly (fine grid 0.025 m)"),
]:
    sc = ax.scatter(grid["lon"], grid["lat"], c=grid["val"], cmap="magma", s=s, marker="s", vmin=vmin, vmax=vmax)
    ax.scatter(objs["Longitude [m]"], objs["Latitude [m]"], color="green", marker="*", s=90)
    ax.set_title(title, fontsize=18, loc="left")
    ax.tick_params(labelsize=14)
    ax.grid(True, linestyle="--", alpha=0.3)
    cbar = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=14)
    cbar.set_label("Detrended Field [nT]", fontsize=16)

min_lon, min_lat, max_lon, max_lat = polygon.bounds
axes[1].set_xlim(min_lon, max_lon)
axes[1].set_ylim(min_lat, max_lat)

fig.text(0.5, 0.04, "Longitude [m]", ha="center", fontsize=18)
fig.text(0.04, 0.5, "Latitude [m]", va="center", rotation="vertical", fontsize=18)
plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.95])
plt.show()